# PaliGemma Evaluation for Indoor Navigation

**TFG Alberto Gomez**

This notebook evaluates different PaliGemma models on indoor navigation images.

## Models to evaluate:
- google/paligemma-3b-pt-224
- google/paligemma-3b-pt-448  
- google/paligemma-3b-mix-224
- google/paligemma-3b-mix-448

## Metrics:
- BLEU score
- METEOR score
- BERTScore

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes
!pip install -q sacrebleu nltk bert-score
!pip install -q pillow pandas tqdm

In [ ]:
# Login to Hugging Face (required for PaliGemma)
from huggingface_hub import notebook_login
notebook_login()

## 2. Clone Repository and Load Data

In [ ]:
# Clone your repository
!git clone https://github.com/AlbertoGomezS/tfg-AlbertoGomez.git
%cd tfg-AlbertoGomez

In [ ]:
# Load dataset
import os
import json
import torch
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# NLTK setup
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Load dataset metadata
with open('data/dataset.json', 'r') as f:
    dataset = json.load(f)

print(f"Dataset: {dataset['dataset_info']['name']}")
print(f"Total images: {dataset['dataset_info']['total_images']}")
print(f"Categories: {dataset['dataset_info']['categories']}")

## 3. Prepare Samples and Define Functions

In [ ]:
# Prepare evaluation samples
def prepare_samples(dataset):
    samples = []
    for img_data in dataset['images']:
        if img_data.get('suggested_questions') and img_data.get('expected_answer'):
            samples.append({
                'id': img_data['id'],
                'image_path': os.path.join('data', img_data['filename']),
                'category': img_data['category'],
                'question': img_data['suggested_questions'][0],
                'expected_answer': img_data['expected_answer'],
            })
    return samples

samples = prepare_samples(dataset)
print(f"Prepared {len(samples)} samples")
print(f"\nExample: {samples[0]['question']}")

In [ ]:
# Model and inference functions
def load_paligemma(model_id):
    print(f"Loading {model_id}...")
    processor = AutoProcessor.from_pretrained(model_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="auto"
    )
    model.eval()
    print("Model loaded!")
    return model, processor

def run_inference(model, processor, image_path, question):
    image = Image.open(image_path).convert("RGB")
    prompt = f"<image>\n{question}"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device, torch.bfloat16)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generated_text = processor.decode(output[0], skip_special_tokens=True)
    answer = generated_text.split(question)[-1].strip()
    return answer

def compute_metrics(predictions, references):
    from sacrebleu.metrics import BLEU
    from nltk.translate.meteor_score import meteor_score
    from nltk.tokenize import word_tokenize
    from bert_score import score as bert_score
    
    # BLEU
    bleu = BLEU()
    refs = [[ref] for ref in references]
    bleu_score = bleu.corpus_score(predictions, refs).score
    
    # METEOR
    meteor_scores = [meteor_score([word_tokenize(ref)], word_tokenize(pred)) 
                    for pred, ref in zip(predictions, references)]
    meteor_avg = sum(meteor_scores) / len(meteor_scores) * 100
    
    # BERTScore
    P, R, F1 = bert_score(predictions, references, lang="en", verbose=False)
    
    return {
        'BLEU': bleu_score,
        'METEOR': meteor_avg,
        'BERTScore_F1': F1.mean().item() * 100
    }

## 4. Run Evaluation

In [ ]:
# Models to evaluate
MODELS = [
    "google/paligemma-3b-pt-224",
    "google/paligemma-3b-pt-448",
    "google/paligemma-3b-mix-224", 
    "google/paligemma-3b-mix-448",
]

all_results = []
all_predictions = {}

for model_id in MODELS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_id}")
    print(f"{'='*60}")
    
    model, processor = load_paligemma(model_id)
    
    predictions, references, detailed = [], [], []
    
    for sample in tqdm(samples, desc="Processing"):
        try:
            pred = run_inference(model, processor, sample['image_path'], sample['question'])
        except Exception as e:
            print(f"Error: {e}")
            pred = ""
        
        predictions.append(pred)
        references.append(sample['expected_answer'])
        detailed.append({
            'category': sample['category'],
            'question': sample['question'],
            'expected': sample['expected_answer'],
            'predicted': pred
        })
    
    metrics = compute_metrics(predictions, references)
    metrics['Model'] = model_id.split('/')[-1]
    all_results.append(metrics)
    all_predictions[model_id] = detailed
    
    print(f"BLEU: {metrics['BLEU']:.2f} | METEOR: {metrics['METEOR']:.2f} | BERTScore: {metrics['BERTScore_F1']:.2f}")
    
    del model, processor
    torch.cuda.empty_cache()

In [ ]:
# Display results
results_df = pd.DataFrame(all_results)[['Model', 'BLEU', 'METEOR', 'BERTScore_F1']]
print("\n" + "="*60)
print("MODEL COMPARISON RESULTS")
print("="*60)
display(results_df)

# Find best model
best_idx = results_df['BERTScore_F1'].idxmax()
print(f"\n✓ Best model: {results_df.loc[best_idx, 'Model']}")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(results_df))
width = 0.25

ax.bar([i - width for i in x], results_df['BLEU'], width, label='BLEU', color='#2ecc71')
ax.bar(x, results_df['METEOR'], width, label='METEOR', color='#3498db')
ax.bar([i + width for i in x], results_df['BERTScore_F1'], width, label='BERTScore F1', color='#e74c3c')

ax.set_ylabel('Score')
ax.set_title('PaliGemma Model Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 100)

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/model_comparison.png', dpi=150)
plt.show()

## 6. Save and Commit Results

In [ ]:
# Save results
results_df.to_csv('results/model_comparison.csv', index=False)
with open('results/detailed_results.json', 'w') as f:
    json.dump(all_predictions, f, indent=2)
print("Results saved!")

# Commit to GitHub
!git add results/
!git commit -m "Add PaliGemma evaluation results"
!git push